# Stress Text EDA and Preprocessing

This notebook prepares the Dreaddit dataset for downstream modeling while keeping exploratory checks transparent and reproducible.


## 1) Setup
Load dependencies and required NLTK resources.


In [1]:
import os
import re
from collections import Counter
from itertools import chain

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


In [2]:
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(resource)

print("NLTK resources are ready.")


NLTK resources are ready.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhuan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bhuan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhuan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhuan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 2) Load raw data
Read Dreaddit train/test data and inspect ADHD reference files.


In [3]:
DREADDIT_TRAIN_PATH = "data/dreaddit/dreaddit-train.csv"
DREADDIT_TEST_PATH = "data/dreaddit/dreaddit-test.csv"
ADHD_DIR = "data/adhd"

train = pd.read_csv(DREADDIT_TRAIN_PATH)
test = pd.read_csv(DREADDIT_TEST_PATH)
df_dreaddit = pd.concat([train, test], ignore_index=True)

print("Dreaddit loaded shape:", df_dreaddit.shape)
print("ADHD folder contents:", os.listdir(ADHD_DIR))


Dreaddit loaded shape: (3553, 116)
ADHD folder contents: ['ADHD-comment.csv', 'ADHD.csv', 'adhdwomen-comment.csv', 'adhdwomen.csv']


In [4]:
adhd1 = pd.read_csv("data/adhd/ADHD.csv")
adhd2 = pd.read_csv("data/adhd/ADHD-comment.csv")
adhd3 = pd.read_csv("data/adhd/adhdwomen.csv")
adhd4 = pd.read_csv("data/adhd/adhdwomen-comment.csv")

print("ADHD.csv columns:", adhd1.columns.tolist())
print("ADHD-comment.csv columns:", adhd2.columns.tolist())


C:\Users\bhuan\AppData\Local\Temp\ipykernel_7588\3705656614.py:1: DtypeWarning: Columns (0: score, 1: created_utc) have mixed types. Specify dtype option on import or set low_memory=False.
  adhd1 = pd.read_csv("data/adhd/ADHD.csv")


ADHD.csv columns: ['title', 'selftext', 'score', 'id', 'url', 'num_comments', 'created_utc', 'created_datetime']
ADHD-comment.csv columns: ['body', 'id', 'score', 'created_utc', 'created_datetime']


## 3) Column selection and text normalization


In [5]:
dreaddit_cols = [
    "subreddit",
    "text",
    "label",
    "confidence",
    "social_karma",
    "lex_liwc_WC",
    "lex_liwc_Tone",
    "lex_liwc_Analytic",
    "lex_liwc_Authentic",
    "lex_liwc_Clout",
    "lex_liwc_affect",
    "lex_liwc_anx",
    "lex_liwc_anger",
    "lex_liwc_sad",
]
df_dreaddit = df_dreaddit[dreaddit_cols]

print("Selected Dreaddit shape:", df_dreaddit.shape)
df_dreaddit.head(2)


Selected Dreaddit shape: (3553, 14)


,subreddit,text,label,confidence,social_karma,lex_liwc_WC,lex_liwc_Tone,lex_liwc_Analytic,lex_liwc_Authentic,lex_liwc_Clout,lex_liwc_affect,lex_liwc_anx,lex_liwc_anger,lex_liwc_sad
0,ptsd,"He said he had not felt that way before, sugge...",1,0.8,5,116,1.00,72.64,89.26,15.04,8.62,0.86,2.59,3.45
1,assistance,"Hey there r/assistance, Not sure if this is th...",0,1.0,4,109,98.18,79.08,56.75,76.85,5.50,0.00,0.00,0.00


In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\[deleted\]|\[removed\]", "", text)
    text = re.sub(r"&amp;|&lt;|&gt;", "", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"[^a-z0-9\s\.\,\!\?]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_dreaddit["text_clean"] = df_dreaddit["text"].apply(clean_text)
df_dreaddit = df_dreaddit[df_dreaddit["text_clean"].str.len() > 10]
df_dreaddit.drop_duplicates(subset="text_clean", inplace=True)
df_dreaddit.dropna(subset=["label"], inplace=True)
df_dreaddit["label_str"] = df_dreaddit["label"].map({1: "stressed", 0: "not_stressed"})

print("After text cleaning:", df_dreaddit.shape)


After text cleaning: (3530, 16)


## 4) Tokenization and feature engineering


In [7]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token.isalpha()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

df_dreaddit["tokens"] = df_dreaddit["text_clean"].apply(preprocess)
df_dreaddit["text_processed"] = df_dreaddit["tokens"].apply(lambda items: " ".join(items))

df_dreaddit["word_count"] = df_dreaddit["tokens"].apply(len)
df_dreaddit["sentence_count"] = df_dreaddit["text_clean"].apply(
    lambda value: len(re.split(r"[.!?]+", value))
)
df_dreaddit["avg_word_length"] = df_dreaddit["tokens"].apply(
    lambda items: round(np.mean([len(word) for word in items]), 2) if items else 0
)

df_dreaddit[["text_clean", "word_count", "sentence_count", "avg_word_length", "label_str"]].head(3)


,text_clean,word_count,sentence_count,avg_word_length,label_str
0,"he said he had not felt that way before, sugge...",52,9,5.46,stressed
1,"hey there rassistance, not sure if this is the...",59,5,5.69,not_stressed
2,my mom then hit me with the newspaper and it s...,68,6,5.74,stressed


In [8]:
df_dreaddit.to_csv("data/dreaddit_clean.csv", index=False)
print("Saved cleaned data to data/dreaddit_clean.csv")
print("Current shape:", df_dreaddit.shape)


Saved cleaned data to data/dreaddit_clean.csv
Current shape: (3530, 21)


## 5) Sanity checks and dataset statistics


In [9]:
missing = df_dreaddit.isnull().sum()
print(missing[missing > 0])
print("Total missing values:", missing.sum())

df_dreaddit.drop_duplicates(subset="text_processed", inplace=True)
print("Duplicate processed texts:", df_dreaddit.duplicated(subset=["text_processed"]).sum())

list_like_columns = [
    col for col in df_dreaddit.columns
    if df_dreaddit[col].apply(lambda value: isinstance(value, list)).any()
]
print("Columns containing list values:", list_like_columns)

print(df_dreaddit["label_str"].value_counts())
print()
print(df_dreaddit["label_str"].value_counts(normalize=True) * 100)


Series([], dtype: int64)
Total missing values: 0
Duplicate processed texts: 0
Columns containing list values: ['tokens']
label_str
stressed        1848
not_stressed    1681
Name: count, dtype: int64

label_str
stressed        52.366109
not_stressed    47.633891
Name: proportion, dtype: float64


In [10]:
df_dreaddit = df_dreaddit[df_dreaddit["word_count"] >= 5]
df_dreaddit = df_dreaddit[df_dreaddit["word_count"] <= 500]

print("Shape after word-count bounds:", df_dreaddit.shape)
print(df_dreaddit["word_count"].describe())

empty_processed = (df_dreaddit["text_processed"].str.strip() == "").sum()
print("Empty processed texts:", empty_processed)


Shape after word-count bounds: (3528, 21)
count    3528.000000
mean       40.036565
std        14.917031
min         6.000000
25%        30.000000
50%        38.000000
75%        47.000000
max       148.000000
Name: word_count, dtype: float64
Empty processed texts: 0


In [11]:
df_dreaddit.groupby("label_str")["word_count"].describe()


,count,mean,std,min,25%,50%,75%,max
label_str,,,,,,,,
not_stressed,1680.0,38.773810,13.624988,6.0,30.0,37.0,46.0,129.0
stressed,1848.0,41.184524,15.918309,8.0,31.0,38.0,49.0,148.0


In [12]:
vocab = set(chain.from_iterable(df_dreaddit["tokens"]))
print("Vocabulary size:", len(vocab))

all_words = Counter(chain.from_iterable(df_dreaddit["tokens"]))
all_words.most_common(20)


Vocabulary size: 11467


[('im', 2233),
 ('like', 1518),
 ('dont', 1183),
 ('get', 1174),
 ('time', 1171),
 ('know', 1128),
 ('feel', 1084),
 ('would', 935),
 ('ive', 883),
 ('year', 849),
 ('really', 811),
 ('want', 791),
 ('even', 754),
 ('thing', 726),
 ('one', 725),
 ('go', 694),
 ('help', 694),
 ('day', 686),
 ('friend', 650),
 ('people', 594)]

In [13]:
numeric_cols = [
    "word_count",
    "sentence_count",
    "avg_word_length",
    "social_karma",
    "confidence",
    "lex_liwc_WC",
    "lex_liwc_Tone",
    "lex_liwc_Analytic",
    "lex_liwc_Authentic",
    "lex_liwc_Clout",
    "lex_liwc_affect",
    "lex_liwc_anx",
    "lex_liwc_anger",
    "lex_liwc_sad",
]

df_dreaddit[numeric_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
word_count,3528.0,40.036565,14.917031,6.00,30.0000,38.000,47.0000,148.00
sentence_count,3528.0,5.875567,0.886910,1.00,6.0000,6.000,6.0000,19.00
avg_word_length,3528.0,5.540116,0.524460,3.89,5.1800,5.500,5.8400,9.26
social_karma,3528.0,19.612245,87.564293,0.00,2.0000,5.000,10.0000,1687.00
confidence,3528.0,0.790187,0.218676,0.00,0.6000,0.800,1.0000,1.00
lex_liwc_WC,3528.0,85.947562,32.078491,8.00,65.0000,81.000,101.0000,310.00
lex_liwc_Tone,3528.0,33.018747,35.079277,1.00,1.3600,16.455,60.4775,99.00
lex_liwc_Analytic,3528.0,35.000173,26.382196,1.00,12.3500,29.370,54.3825,99.00
lex_liwc_Authentic,3528.0,67.815091,32.557862,1.00,43.3700,81.480,96.4000,99.00
lex_liwc_Clout,3528.0,40.239702,31.331395,1.00,11.7725,33.020,67.6800,99.00


In [14]:
df_dreaddit.dtypes


subreddit                 str
text                      str
label                   int64
confidence            float64
social_karma            int64
lex_liwc_WC             int64
lex_liwc_Tone         float64
lex_liwc_Analytic     float64
lex_liwc_Authentic    float64
lex_liwc_Clout        float64
lex_liwc_affect       float64
lex_liwc_anx          float64
lex_liwc_anger        float64
lex_liwc_sad          float64
text_clean                str
label_str                 str
tokens                 object
text_processed            str
word_count              int64
sentence_count          int64
avg_word_length       float64
dtype: object

## 6) Final export


In [15]:
df_dreaddit.reset_index(drop=True, inplace=True)
df_dreaddit.to_csv("data/dreaddit_preprocessed.csv", index=False)
print("Saved successfully to data/dreaddit_preprocessed.csv")


Saved successfully to data/dreaddit_preprocessed.csv
